In [6]:
import sys
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [7]:
pip install psutil polars pyarrow tqdm

Note: you may need to restart the kernel to use updated packages.


# 🏦 AML Mule Detection — UPGRADED Memory-Safe Pipeline
### New: Patterns A-F + Series/Parallel Ensemble + Autoencoder + Cluster Isolation Forest
### ⚠️ Run ONE cell at a time. Never skip memory clear cells.
---

## ⚙️ CELL 1: Mount Drive + Install Packages

In [8]:
# ── LOCAL JUPYTER SETUP (replaces Google Colab drive mount) ──
import os

# ⚠️ UPDATE THIS to the folder where your data files are stored
DATA_DIR = r"C:\Users\DHANUSH A G\Downloads\aml_data"  # Windows example
# DATA_DIR = "/home/yourname/aml_data"        # Linux/Mac example

os.chdir(DATA_DIR)
print("Working dir:", os.getcwd())
print("Files found:", sorted(os.listdir(".")))


Working dir: C:\Users\DHANUSH A G\Downloads\aml_data
Files found: ['README.md', 'accounts-additional.parquet', 'accounts.parquet', 'branch.parquet', 'customer_account_linkage.parquet', 'customers.parquet', 'demographics.parquet', 'product_details.parquet', 'test_accounts.parquet', 'train_labels.parquet', 'transactions', 'transactions_additional']


In [9]:
# ── Install required packages (run once in terminal if preferred) ──
import subprocess, sys
pkgs = ["polars", "pyarrow", "lightgbm", "xgboost", "scikit-learn", "shap", "psutil"]
for pkg in pkgs:
    try:
        __import__(pkg.replace("-","_"))
        print(f"  ✅ {pkg} already installed")
    except ImportError:
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✅ All packages ready")


  ✅ polars already installed
  ✅ pyarrow already installed
  ✅ lightgbm already installed
  ✅ xgboost already installed
  Installing scikit-learn...


C:\Users\DHANUSH A G\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✅ shap already installed
  ✅ psutil already installed
✅ All packages ready


## ⚙️ CELL 2: Imports + Memory Helper

In [10]:
import pandas as pd
import numpy as np
import polars as pl
import gc, ctypes, os, warnings, platform
from glob import glob
from collections import defaultdict
warnings.filterwarnings('ignore')

REFERENCE_DATE = pd.Timestamp('2025-07-01')
DATA_DIR = os.getcwd()  # already set in Cell 1

def clear_memory():
    gc.collect()
    if platform.system() == 'Linux':
        try: ctypes.CDLL('libc.so.6').malloc_trim(0)
        except: pass
    # Windows: gc.collect() is sufficient

def show_ram():
    import psutil
    r = psutil.virtual_memory()
    print(f'RAM: {r.used/1e9:.1f}GB used / {r.total/1e9:.1f}GB total / {r.available/1e9:.1f}GB FREE')

show_ram()
print('✅ Ready')

RAM: 11.6GB used / 17.0GB total / 5.4GB FREE
✅ Ready


In [11]:
# ── LAPTOP OPTIMISATION SETTINGS (16GB RAM) ──
import psutil, polars as pl

total_ram_gb = psutil.virtual_memory().total / 1e9
print(f"Total RAM: {total_ram_gb:.1f} GB")

# Polars: limit thread count so RAM-heavy ops don't spike all 8 cores at once
pl.Config.set_streaming_chunk_size(50_000)   # smaller chunks in streaming ops

# Recommended: close Chrome/other apps before running Steps 2B and 2C
# Those steps use ~8-10 GB peak RAM

# If you hit MemoryError, set this flag to True to use even smaller batch sizes:
LOW_MEM_MODE = False   # set True if you have <12 GB free

if total_ram_gb < 14:
    LOW_MEM_MODE = True
    print("⚠️  Low RAM detected — enabling LOW_MEM_MODE")
else:
    print("✅ RAM looks sufficient — LOW_MEM_MODE off")

show_ram()


Total RAM: 17.0 GB
✅ RAM looks sufficient — LOW_MEM_MODE off
RAM: 11.6GB used / 17.0GB total / 5.4GB FREE


---
# 🧹 MEMORY CLEAR — Run this between every step

In [12]:
to_delete = [
    'customers','accounts','demographics','acc_additional',
    'branch','linkage','product','train_labels','test_accounts',
    'acc_feat','scheme_feat','cust_feat','static_features',
    'txn_sample','df','chunk','agg','geo_agg','txn_id_map',
    'records','geo_records','features','train_feat','test_feat',
    'X_tr','X_te','txn_features','geo_features','clean_labels',
    'extra_features','net_features','vel_features',
    'lgb_oof','xgb_oof','meta_oof','oof_stack','A',
    'CP_CREDIT','CP_DEBIT','CHANNELS','MONTHLY','MONTHLY_C',
    'FIRST_TS','LAST_TS','RECENT_TXN','VEL','GEO','IP_MAP'
]
import gc, ctypes
for v in to_delete:
    if v in dir():
        try: exec(f'del {v}')
        except: pass
gc.collect()
try: ctypes.CDLL('libc.so.6').malloc_trim(0)
except: pass
show_ram()
print('✅ Memory cleared')

RAM: 11.6GB used / 17.0GB total / 5.4GB FREE
✅ Memory cleared


---
# 📊 STEP 1: EDA
Loads files one at a time, deletes after use.

In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

train_labels  = pd.read_parquet('train_labels.parquet',
    columns=['account_id','is_mule','alert_reason','flagged_by_branch'])
test_accounts = pd.read_parquet('test_accounts.parquet', columns=['account_id'])
show_ram()

mule_counts = train_labels['is_mule'].value_counts()
print(f'Legitimate: {mule_counts[0]} ({mule_counts[0]/len(train_labels)*100:.1f}%)')
print(f'Mule:       {mule_counts[1]} ({mule_counts[1]/len(train_labels)*100:.1f}%)')
print('\nTop alert reasons:')
print(train_labels[train_labels['is_mule']==1]['alert_reason'].value_counts().head(8))

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].bar(['Legitimate','Mule'],[mule_counts[0],mule_counts[1]],color=['steelblue','crimson'])
axes[0].set_title('Label Distribution')
al = train_labels[train_labels['is_mule']==1]['alert_reason'].value_counts().head(8)
axes[1].barh(al.index, al.values, color='coral')
axes[1].set_title('Top Alert Reasons')
plt.tight_layout(); plt.savefig('plot_labels.png',dpi=100); plt.show(); plt.close()
del al; clear_memory()
print('✅ EDA done')

RAM: 11.6GB used / 17.0GB total / 5.4GB FREE
Legitimate: 93408 (97.2%)
Mule:       2683 (2.8%)

Top alert reasons:
alert_reason
Routine Investigation                       705
Dormant Account Reactivation                188
Rapid Movement of Funds                     177
MCC-Amount Distribution Anomaly             150
Structuring Transactions Below Threshold    146
Geographic Anomaly Detected                 144
Unusual Fund Flow Pattern                   141
Income-Transaction Mismatch                 134
Name: count, dtype: int64
✅ EDA done


In [14]:
# Account + scheme EDA
acc_eda = pd.read_parquet('accounts.parquet',
    columns=['account_id','account_status','product_family','kyc_compliant','avg_balance','freeze_date'])
acc_labeled = acc_eda.merge(train_labels[['account_id','is_mule']], on='account_id', how='left')
del acc_eda; clear_memory()

print('Mule rate by product family:')
print(acc_labeled.groupby('product_family')['is_mule'].mean().sort_values(ascending=False).round(4))
print('\nMule rate by KYC:')
print(acc_labeled.groupby('kyc_compliant')['is_mule'].mean().round(4))
del acc_labeled; clear_memory()

# Sample transactions EDA - ONE file only
parts = sorted(glob('transactions/batch-1/part_*.parquet'))
txn_s = pd.read_parquet(parts[0])
txn_s['transaction_timestamp'] = pd.to_datetime(txn_s['transaction_timestamp'])
print(f'\nSample txn shape: {txn_s.shape}')
print('Channels:', txn_s['channel'].value_counts().head(8).to_dict())
del txn_s; clear_memory()
show_ram()
print('✅ STEP 1 COMPLETE')

Mule rate by product family:
product_family
K    0.0287
S    0.0276
O    0.0275
Name: is_mule, dtype: float64

Mule rate by KYC:
kyc_compliant
N    0.0254
Y    0.0282
Name: is_mule, dtype: float64

Sample txn shape: (1000000, 8)
Channels: {'UPC': 379934, 'UPD': 359340, 'IPM': 38212, 'P2A': 27975, 'STD': 23714, 'END': 22892, 'NTD': 15703, 'FTD': 10459}
RAM: 11.8GB used / 17.0GB total / 5.2GB FREE
✅ STEP 1 COMPLETE


---
# 🔧 STEP 2A: Static Feature Engineering
Includes Pattern E (Account Opening Clusters)

In [15]:
# clear first
for v in ['train_labels','test_accounts']:
    if v in dir():
        try: exec(f'del {v}')
        except: pass
clear_memory()

train_labels  = pd.read_parquet('train_labels.parquet', columns=['account_id','is_mule'])
test_accounts = pd.read_parquet('test_accounts.parquet', columns=['account_id'])
all_accounts  = pd.concat([train_labels[['account_id']], test_accounts[['account_id']]],
                           ignore_index=True).drop_duplicates()
print(f'Total accounts: {len(all_accounts):,}')
show_ram()

Total accounts: 160,153
RAM: 11.8GB used / 17.0GB total / 5.2GB FREE


In [16]:
# ── ACCOUNT FEATURES ──
print('Building account features...')
acc_cols = [
    'account_id','account_status','avg_balance','product_family',
    'nomination_flag','kyc_compliant','rural_branch','cheque_availed',
    'num_chequebooks','account_opening_date','last_mobile_update_date',
    'last_kyc_date','freeze_date','unfreeze_date','branch_code',
    'monthly_avg_balance','quarterly_avg_balance','daily_avg_balance'
]
acc = pd.read_parquet('accounts.parquet', columns=acc_cols)
for col in ['account_opening_date','last_mobile_update_date','last_kyc_date','freeze_date','unfreeze_date']:
    acc[col] = pd.to_datetime(acc[col])

acc['account_age_days']           = (REFERENCE_DATE - acc['account_opening_date']).dt.days
acc['days_since_mobile_update']   = (REFERENCE_DATE - acc['last_mobile_update_date']).dt.days
acc['days_since_kyc']             = (REFERENCE_DATE - acc['last_kyc_date']).dt.days
acc['is_frozen']                  = acc['freeze_date'].notna().astype(np.int8)
acc['was_unfrozen']               = acc['unfreeze_date'].notna().astype(np.int8)
acc['freeze_duration_days']       = (acc['unfreeze_date'] - acc['freeze_date']).dt.days.fillna(0)
acc['balance_variance']           = acc[['monthly_avg_balance','quarterly_avg_balance','daily_avg_balance']].std(axis=1)
acc['balance_diff_monthly_daily'] = (acc['monthly_avg_balance'] - acc['daily_avg_balance']).abs()
acc['is_kyc_noncompliant']        = (acc['kyc_compliant']=='N').astype(np.int8)
acc['new_account_flag']           = (acc['account_age_days'] < 180).astype(np.int8)
acc['recent_mobile_change']       = (acc['days_since_mobile_update'] < 90).astype(np.int8)
acc['product_family_enc']         = acc['product_family'].map({'S':0,'K':1,'O':2}).astype(np.int8)
acc['account_status_enc']         = (acc['account_status']=='frozen').astype(np.int8)
acc['nomination_flag_enc']        = (acc['nomination_flag']=='Y').astype(np.int8)
acc['rural_branch_enc']           = (acc['rural_branch']=='Y').astype(np.int8)
acc['cheque_availed_enc']         = (acc['cheque_availed']=='Y').astype(np.int8)

# ── PATTERN E: Account Opening Cluster ──
# Accounts opened same branch same week = batch recruitment signal
acc['open_week'] = acc['account_opening_date'].dt.to_period('W').astype(str)
branch_week_counts = acc.groupby(['branch_code','open_week'])['account_id'].transform('count')
acc['accounts_opened_same_branch_week'] = branch_week_counts.astype(np.int16)
acc['kyc_same_day_as_opening'] = (acc['last_kyc_date'].dt.date == acc['account_opening_date'].dt.date).astype(np.int8)

keep = [
    'account_id','branch_code','account_age_days','days_since_mobile_update',
    'days_since_kyc','is_frozen','was_unfrozen','freeze_duration_days',
    'avg_balance','monthly_avg_balance','quarterly_avg_balance','daily_avg_balance',
    'balance_variance','balance_diff_monthly_daily','is_kyc_noncompliant',
    'new_account_flag','recent_mobile_change','product_family_enc',
    'account_status_enc','nomination_flag_enc','rural_branch_enc',
    'cheque_availed_enc','num_chequebooks',
    'accounts_opened_same_branch_week','kyc_same_day_as_opening'
]
acc_feat = acc[keep].copy()
del acc; clear_memory()
print(f'Account features: {acc_feat.shape}')
show_ram()

Building account features...
Account features: (160153, 25)
RAM: 11.9GB used / 17.0GB total / 5.1GB FREE


In [17]:
# ── BRANCH FEATURES ──
print('Building branch features...')
branch = pd.read_parquet('branch.parquet',
    columns=['branch_code','branch_employee_count','branch_turnover','branch_asset_size','branch_type'])
branch['branch_type_enc'] = branch['branch_type'].map({'urban':0,'semi-urban':1,'rural':2}).astype(np.int8)

acc_br = pd.read_parquet('accounts.parquet', columns=['account_id','branch_code'])
mule_by_br = acc_br.merge(train_labels, on='account_id', how='left')
br_stats = mule_by_br.groupby('branch_code')['is_mule'].agg(
    branch_mule_rate='mean', branch_mule_count='sum', branch_account_count='count').reset_index()
del acc_br, mule_by_br; clear_memory()

branch = branch.merge(br_stats, on='branch_code', how='left').drop('branch_type', axis=1)
del br_stats; clear_memory()

acc_feat = acc_feat.merge(branch, on='branch_code', how='left').drop('branch_code', axis=1)
del branch; clear_memory()
print(f'After branch merge: {acc_feat.shape}')
show_ram()

Building branch features...
After branch merge: (160153, 31)
RAM: 11.9GB used / 17.0GB total / 5.1GB FREE


In [18]:
# ── SCHEME FEATURES ──
print('Building scheme features...')
scheme = pd.read_parquet('accounts-additional.parquet', columns=['account_id','scheme_code'])
scheme_dummies = pd.get_dummies(scheme['scheme_code'], prefix='scheme').astype(np.int8)
scheme_feat = pd.concat([scheme['account_id'], scheme_dummies], axis=1)
del scheme, scheme_dummies; clear_memory()
print(f'Scheme features: {scheme_feat.shape}')

Building scheme features...
Scheme features: (160153, 8)


In [19]:
# ── CUSTOMER + DEMOGRAPHICS FEATURES ──
print('Building customer features...')
cust = pd.read_parquet('customers.parquet', columns=[
    'customer_id','date_of_birth','relationship_start_date',
    'pan_available','aadhaar_available','passport_available',
    'mobile_banking_flag','internet_banking_flag','atm_card_flag',
    'demat_flag','credit_card_flag','fastag_flag','customer_pin','permanent_pin'])
cust['date_of_birth']           = pd.to_datetime(cust['date_of_birth'])
cust['relationship_start_date'] = pd.to_datetime(cust['relationship_start_date'])
cust['customer_age']            = (REFERENCE_DATE - cust['date_of_birth']).dt.days / 365.25
cust['relationship_age_days']   = (REFERENCE_DATE - cust['relationship_start_date']).dt.days
for f in ['pan_available','aadhaar_available','passport_available',
          'mobile_banking_flag','internet_banking_flag','atm_card_flag',
          'demat_flag','credit_card_flag','fastag_flag']:
    cust[f+'_enc'] = (cust[f]=='Y').astype(np.int8)
cust['kyc_score']            = cust[['pan_available_enc','aadhaar_available_enc','passport_available_enc']].sum(axis=1).astype(np.int8)
cust['banking_product_count']= cust[['mobile_banking_flag_enc','internet_banking_flag_enc','atm_card_flag_enc',
                                      'demat_flag_enc','credit_card_flag_enc','fastag_flag_enc']].sum(axis=1).astype(np.int8)
cust['pin_mismatch']         = (cust['customer_pin'] != cust['permanent_pin']).astype(np.int8)
cust = cust[['customer_id','customer_age','relationship_age_days','kyc_score','banking_product_count',
             'pin_mismatch','pan_available_enc','aadhaar_available_enc','passport_available_enc']]

demo = pd.read_parquet('demographics.parquet', columns=[
    'customer_id','gender','address_last_update_date','passbook_last_update_date',
    'nri_flag','joint_account_flag'])
demo['address_last_update_date']  = pd.to_datetime(demo['address_last_update_date'])
demo['passbook_last_update_date'] = pd.to_datetime(demo['passbook_last_update_date'])
demo['days_since_address_update'] = (REFERENCE_DATE - demo['address_last_update_date']).dt.days
demo['days_since_passbook_update']= (REFERENCE_DATE - demo['passbook_last_update_date']).dt.days
demo['is_nri']    = (demo['nri_flag']=='Y').astype(np.int8)
demo['is_joint']  = (demo['joint_account_flag']=='Y').astype(np.int8)
demo['gender_enc']= (demo['gender']=='M').astype(np.int8)
demo = demo[['customer_id','days_since_address_update','days_since_passbook_update','is_nri','is_joint','gender_enc']]

prod = pd.read_parquet('product_details.parquet')
link = pd.read_parquet('customer_account_linkage.parquet', columns=['customer_id','account_id'])
num_acc = link.groupby('customer_id')['account_id'].count().reset_index()
num_acc.columns = ['customer_id','num_accounts']

cust_feat = cust.merge(demo,on='customer_id',how='left')                .merge(prod,on='customer_id',how='left')                .merge(num_acc,on='customer_id',how='left')
del cust, demo, prod, num_acc; clear_memory()
print(f'Customer features: {cust_feat.shape}')
show_ram()

Building customer features...
Customer features: (159416, 25)
RAM: 12.0GB used / 17.0GB total / 5.0GB FREE


In [20]:
# ── MERGE ALL STATIC ──
print('Merging all static features...')
static_features = all_accounts.copy()
static_features = static_features.merge(acc_feat, on='account_id', how='left')
del acc_feat; clear_memory()
static_features = static_features.merge(scheme_feat, on='account_id', how='left')
del scheme_feat; clear_memory()
static_features = static_features.merge(link, on='account_id', how='left')
static_features = static_features.merge(cust_feat, on='customer_id', how='left')
static_features = static_features.drop('customer_id', axis=1)
del cust_feat, link; clear_memory()

scheme_cols = [c for c in static_features.columns if c.startswith('scheme_')]
static_features[scheme_cols] = static_features[scheme_cols].fillna(0).astype(np.int8)

static_features.to_parquet('static_features.parquet', index=False)
del static_features; clear_memory()
show_ram()
print('✅ STEP 2A COMPLETE — static_features.parquet saved')

Merging all static features...
RAM: 11.7GB used / 17.0GB total / 5.2GB FREE
✅ STEP 2A COMPLETE — static_features.parquet saved


---
# 🔧 STEP 2B: Transaction Features (Polars + Chunked)
### NEW: Pattern B (Velocity), Pattern C (Behavioral), Pattern D (Balance)
⚠️ Takes 45-90 mins. Keep tab active. DO NOT close.

In [ ]:
# CLEAR FIRST
for v in ['train_labels','test_accounts','all_accounts']:
    if v in dir():
        try: exec(f'del {v}')
        except: pass
clear_memory()
show_ram()

all_parts = sorted(glob('transactions/batch-*/part_*.parquet'))
print(f'Found {len(all_parts)} transaction files')

# ── ACCUMULATORS (all lightweight — no raw data stored) ──
A         = defaultdict(lambda: np.zeros(12, dtype=np.float32))
# 0=total 1=credit_amt 2=debit_amt 3=credit_cnt 4=debit_cnt
# 5=near_thresh 6=round_all 7=large_txn 8=max_txn 9=round_1k 10=round_5k 11=round_50k

CP_CREDIT = defaultdict(set)    # unique credit counterparties
CP_DEBIT  = defaultdict(set)    # unique debit counterparties
CHANNELS  = defaultdict(set)    # unique channels used
MONTHLY   = defaultdict(lambda: defaultdict(int))    # month -> txn count
MONTHLY_C = defaultdict(lambda: defaultdict(float))  # month -> credit amt
FIRST_TS  = {}
LAST_TS   = {}

# Pattern B: Velocity — track daily txn counts (bounded dict)
DAILY_CNT = defaultdict(lambda: defaultdict(int))  # day_key -> count

# Pattern B: Weekend/night signals
WEEKEND_CNT = defaultdict(int)
NIGHT_CNT   = defaultdict(int)   # midnight-6am

# Pattern C: Behavioral — MCC diversity
MCC_SET   = defaultdict(set)

# Pattern D: Balance — track near-zero balance events
# (done in Step 2C from balance_after_transaction)

# Pass-through: keep last 100 txns per account (bounded)
RECENT_TXN = defaultdict(list)

print('Processing with Polars...')
for i, part_path in enumerate(all_parts):
    if i % 30 == 0:
        print(f'  Part {i+1}/{len(all_parts)}...')
        show_ram()

    df = pl.read_parquet(part_path, columns=[
        'account_id','transaction_timestamp','amount',
        'txn_type','channel','counterparty_id','mcc_code'])

    df = df.with_columns([
        pl.col('amount').abs().alias('amount'),
        pl.col('transaction_timestamp').str.to_datetime(format=None, strict=False).alias('ts'),
        (pl.col('transaction_timestamp').str.to_datetime(format=None, strict=False).dt.year().cast(pl.Utf8) + '-' +
         pl.col('transaction_timestamp').str.to_datetime(format=None, strict=False).dt.month().cast(pl.Utf8)).alias('month_key'),
        pl.col('transaction_timestamp').str.to_datetime(format=None, strict=False).dt.date().cast(pl.Utf8).alias('day_key'),
        pl.col('transaction_timestamp').str.to_datetime(format=None, strict=False).dt.weekday().alias('weekday'),
        pl.col('transaction_timestamp').str.to_datetime(format=None, strict=False).dt.hour().alias('hour'),
    ])

    # ── Polars groupby aggregations (vectorized, fast) ──
    agg_b = df.group_by(['account_id','txn_type']).agg([
        pl.col('amount').sum().alias('total_amt'),
        pl.col('amount').count().alias('cnt'),
        pl.col('amount').max().alias('max_amt'),
        ((pl.col('amount')>=40000)&(pl.col('amount')<50000)).sum().alias('near_thresh'),
        ((pl.col('amount')==1000)|(pl.col('amount')==5000)|
         (pl.col('amount')==10000)|(pl.col('amount')==50000)).sum().alias('round_all'),
        (pl.col('amount')==1000).sum().alias('r1k'),
        (pl.col('amount')==5000).sum().alias('r5k'),
        (pl.col('amount')==50000).sum().alias('r50k'),
        (pl.col('amount')>100000).sum().alias('large'),
    ]).to_pandas()

    # Update A accumulator using itertuples (10x faster than iterrows)
    for row in agg_b.itertuples(index=False):
        aid = row.account_id
        is_c = row.txn_type == 'C'
        A[aid][0] += row.cnt
        if is_c:
            A[aid][1] += row.total_amt; A[aid][3] += row.cnt
        else:
            A[aid][2] += row.total_amt; A[aid][4] += row.cnt
        A[aid][5]  += row.near_thresh
        A[aid][6]  += row.round_all
        A[aid][7]  += row.large
        A[aid][9]  += row.r1k
        A[aid][10] += row.r5k
        A[aid][11] += row.r50k
        if row.max_amt > A[aid][8]: A[aid][8] = row.max_amt

    # Counterparties (vectorized groupby then update sets)
    cp = df.filter(pl.col('counterparty_id').is_not_null())           .select(['account_id','txn_type','counterparty_id']).to_pandas()
    cp_c = cp[cp['txn_type']=='C'].groupby('account_id')['counterparty_id'].apply(set)
    cp_d = cp[cp['txn_type']=='D'].groupby('account_id')['counterparty_id'].apply(set)
    for aid, s in cp_c.items(): CP_CREDIT[aid].update(s)
    for aid, s in cp_d.items(): CP_DEBIT[aid].update(s)
    del cp, cp_c, cp_d

    # Channels
    ch = df.select(['account_id','channel']).to_pandas()
    for aid, grp in ch.groupby('account_id'):
        CHANNELS[aid].update(grp['channel'].tolist())
    del ch

    # Monthly
    mo = df.group_by(['account_id','month_key','txn_type']).agg([
        pl.col('amount').count().alias('cnt'),
        pl.col('amount').sum().alias('sum_amt')
    ]).to_pandas()
    for row in mo.itertuples(index=False):
        MONTHLY[row.account_id][row.month_key] += row.cnt
        if row.txn_type == 'C':
            MONTHLY_C[row.account_id][row.month_key] += row.sum_amt
    del mo

    # Timestamps
    ts_agg = df.group_by('account_id').agg([
        pl.col('ts').min().alias('min_ts'),
        pl.col('ts').max().alias('max_ts')
    ]).to_pandas()
    for row in ts_agg.itertuples(index=False):
        aid = row.account_id
        if aid not in FIRST_TS or row.min_ts < FIRST_TS[aid]: FIRST_TS[aid] = row.min_ts
        if aid not in LAST_TS  or row.max_ts > LAST_TS[aid]:  LAST_TS[aid]  = row.max_ts
    del ts_agg

    # Pattern B: Daily velocity (for 24h/7d windows)
    dv = df.group_by(['account_id','day_key']).agg(pl.col('amount').count().alias('cnt')).to_pandas()
    for row in dv.itertuples(index=False):
        DAILY_CNT[row.account_id][row.day_key] += row.cnt
    del dv

    # Pattern B: Weekend + night
    wn = df.select(['account_id','weekday','hour']).to_pandas()
    wn_grp = wn.groupby('account_id')
    for aid, grp in wn_grp:
        WEEKEND_CNT[aid] += (grp['weekday'] >= 5).sum()
        NIGHT_CNT[aid]   += ((grp['hour'] >= 0) & (grp['hour'] < 6)).sum()
    del wn, wn_grp

    # Pattern C: MCC diversity
    mcc = df.filter(pl.col('mcc_code').is_not_null()).select(['account_id','mcc_code']).to_pandas()
    for aid, grp in mcc.groupby('account_id'):
        MCC_SET[aid].update(grp['mcc_code'].tolist())
    del mcc

    # Pass-through (bounded to last 100)
    rec = df.sort('ts').select(['account_id','ts','amount','txn_type']).to_pandas()
    for aid, grp in rec.groupby('account_id'):
        new = list(zip(grp['ts'], grp['amount'], grp['txn_type']))
        RECENT_TXN[aid].extend(new)
        if len(RECENT_TXN[aid]) > 100:
            RECENT_TXN[aid] = RECENT_TXN[aid][-100:]
    del rec

    del df, agg_b
    clear_memory()

print('\n✅ All parts processed')
show_ram()

RAM: 11.7GB used / 17.0GB total / 5.3GB FREE
Found 396 transaction files
Processing with Polars...
  Part 1/396...
RAM: 11.7GB used / 17.0GB total / 5.3GB FREE


In [ ]:
# ── BUILD TRANSACTION FEATURE DATAFRAME ──
print('Computing features from accumulators...')
records = []

for acc_id, a in A.items():
    r = {'account_id': acc_id}
    total = max(a[0], 1)

    # Basic
    r['total_txns']           = float(a[0])
    r['total_credit_amt']     = float(a[1])
    r['total_debit_amt']      = float(a[2])
    r['credit_count']         = float(a[3])
    r['debit_count']          = float(a[4])
    r['avg_txn_amount']       = (a[1]+a[2]) / total
    r['credit_debit_ratio']   = float(a[1]) / max(float(a[2]), 1)
    r['credit_debit_balance'] = abs(float(a[1]) - float(a[2]))

    # Structuring
    r['near_threshold_count'] = float(a[5])
    r['structuring_ratio']    = float(a[5]) / total

    # Round amounts
    r['round_amount_count']   = float(a[6])
    r['round_amount_ratio']   = float(a[6]) / total
    r['round_1k_count']       = float(a[9])
    r['round_5k_count']       = float(a[10])
    r['round_50k_count']      = float(a[11])

    # High value
    r['large_txn_count']      = float(a[7])
    r['large_txn_ratio']      = float(a[7]) / total
    r['max_single_txn']       = float(a[8])

    # Counterparties (Pattern A proxy + Pattern 4)
    cc = len(CP_CREDIT.get(acc_id, set()))
    cd = len(CP_DEBIT.get(acc_id, set()))
    r['unique_credit_counterparties'] = cc
    r['unique_debit_counterparties']  = cd
    r['fan_in_score']                 = cc / max(cd, 1)
    r['counterparty_reuse_rate']      = 1 - (cc / max(float(a[3]), 1))  # Pattern C

    # Channels
    ch = CHANNELS.get(acc_id, set())
    r['channel_count'] = len(ch)

    # Temporal span
    ft = FIRST_TS.get(acc_id)
    lt = LAST_TS.get(acc_id)
    if ft and lt:
        try:
            span = (lt - ft).days
        except:
            span = 0
        r['active_span_days'] = span
        r['txn_frequency']    = float(a[0]) / max(span, 1)
    else:
        r['active_span_days'] = 0
        r['txn_frequency']    = 0

    # Monthly stats
    mc = MONTHLY.get(acc_id, {})
    if mc:
        counts  = list(mc.values())
        mean_c  = np.mean(counts)
        r['monthly_txn_mean']     = mean_c
        r['monthly_txn_std']      = np.std(counts)
        r['monthly_txn_max']      = max(counts)
        r['monthly_txn_cv']       = np.std(counts) / max(mean_c, 1)
        r['burst_ratio']          = max(counts) / max(mean_c, 1)
        active_months = len([c for c in counts if c > 0])
        total_months  = max(r['active_span_days'] // 30, 1)
        r['dormant_months_ratio'] = 1 - (active_months / total_months)
    else:
        r['monthly_txn_mean'] = r['monthly_txn_std'] = r['monthly_txn_max'] = 0
        r['monthly_txn_cv']   = r['burst_ratio'] = r['dormant_months_ratio'] = 0

    # Monthly credit CV
    mc2 = MONTHLY_C.get(acc_id, {})
    if mc2:
        mcv = list(mc2.values())
        r['monthly_credit_cv']  = np.std(mcv) / max(np.mean(mcv), 1)
        r['monthly_credit_max'] = max(mcv)
    else:
        r['monthly_credit_cv']  = 0
        r['monthly_credit_max'] = 0

    # PATTERN B: Velocity features
    daily = DAILY_CNT.get(acc_id, {})
    if daily:
        dcounts = list(daily.values())
        r['velocity_max_24h']   = max(dcounts)
        r['velocity_mean_24h']  = np.mean(dcounts)
        # 7-day velocity: sort days and find max rolling 7-day sum
        sorted_days = sorted(daily.keys())
        max_7d = 0
        for j in range(len(sorted_days)):
            window_sum = sum(daily[d] for k, d in enumerate(sorted_days)
                           if abs(k - j) <= 3)
            if window_sum > max_7d:
                max_7d = window_sum
        r['velocity_max_7d']    = max_7d
        r['velocity_spike_ratio'] = max(dcounts) / max(np.mean(dcounts), 1)
    else:
        r['velocity_max_24h'] = r['velocity_mean_24h'] = 0
        r['velocity_max_7d']  = r['velocity_spike_ratio'] = 0

    # PATTERN B: Weekend + night ratio
    wknd = WEEKEND_CNT.get(acc_id, 0)
    nght = NIGHT_CNT.get(acc_id, 0)
    r['weekend_txn_ratio'] = wknd / total
    r['night_txn_ratio']   = nght / total

    # PATTERN C: MCC diversity
    r['mcc_diversity_score'] = len(MCC_SET.get(acc_id, set()))

    # Pass-through
    recent = sorted(RECENT_TXN.get(acc_id, []), key=lambda x: x[0])
    ptc = 0
    for j in range(len(recent)):
        if recent[j][2] == 'C':
            for k in range(j+1, min(j+10, len(recent))):
                try:
                    diff = (recent[k][0] - recent[j][0]).total_seconds() / 3600
                except:
                    diff = 999
                if diff > 24: break
                if recent[k][2]=='D' and abs(recent[k][1]-recent[j][1])/max(recent[j][1],1)<0.1:
                    ptc += 1; break
    r['pass_through_count'] = ptc
    r['pass_through_ratio'] = ptc / max(float(a[3]), 1)

    records.append(r)

txn_features = pd.DataFrame(records)
txn_features.to_parquet('transaction_features.parquet', index=False)
print(f'Transaction features: {txn_features.shape}')
print(f'New features added: velocity_max_24h, velocity_max_7d, weekend_txn_ratio, night_txn_ratio, mcc_diversity_score, counterparty_reuse_rate')

del A, CP_CREDIT, CP_DEBIT, CHANNELS, MONTHLY, MONTHLY_C
del FIRST_TS, LAST_TS, RECENT_TXN, DAILY_CNT, WEEKEND_CNT, NIGHT_CNT, MCC_SET, records
del txn_features
clear_memory()
show_ram()
print('✅ STEP 2B COMPLETE — transaction_features.parquet saved')

---
# 🔧 STEP 2C: Geo + IP Features + Pattern F (IP Subnet)
### NEW: ip_subnet_sharing, single_ip_high_volume, balance_floor_ratio
⚠️ Takes 30-60 mins.

In [ ]:
show_ram()

# ═══════════════════════════════════════════════════════════════
# STEP 2C: Geo + IP Features  ——  OPTIMIZED v2
#
# ROOT CAUSE of old crash / 3hr+ runtime:
#   1. txn_id_map = {} built by row-iterating ALL 400M transactions
#      into a Python dict  →  killed RAM and took hours
#   2. inner loop `for row in df.iter_rows(named=True)` on geo file
#      →  Python-speed iteration over millions of rows
#
# FIX:
#   • Join transactions_additional ↔ transactions on transaction_id
#     using Polars (vectorized, C-speed)  — no Python dict needed
#   • All aggregations done with Polars group_by → to_pandas()
#   • Only tiny summary DataFrames are iterated in Python
# ═══════════════════════════════════════════════════════════════

from collections import defaultdict, Counter
import numpy as np, pandas as pd, polars as pl, gc, os

# Mobile-update date lookup (small table — fine as dict)
mob = pd.read_parquet('accounts.parquet', columns=['account_id','last_mobile_update_date'])
mob['last_mobile_update_date'] = pd.to_datetime(mob['last_mobile_update_date'])
mob_pl = pl.from_pandas(mob[['account_id','last_mobile_update_date']]).with_columns(
    pl.col('last_mobile_update_date').cast(pl.Datetime)
)
del mob; gc.collect()

txn_parts = sorted(glob('transactions/batch-*/part_*.parquet'))
geo_parts = sorted(glob('transactions_additional/batch-*/part_*.parquet'))

# Map part filename → txn path for pairing (parts are identically named)
txn_by_name = {os.path.basename(p): p for p in txn_parts}

# ── Lightweight accumulators (summaries only, not raw rows) ──
GEO = defaultdict(lambda: {
    'lat_stds': [], 'lon_stds': [],
    'lat_range': 0.0, 'lon_range': 0.0,
    'ips': set(), 'ip_list': [],
    'bals': [],          # list of (std, min, max, near_zero_cnt, cnt) per part
    'ci': 0, 'bi': 0, 'cash': 0, 'post_mob': 0
})

print(f'Processing {len(geo_parts)} geo parts paired with transaction parts...')

for i, geo_p in enumerate(geo_parts):
    if i % 30 == 0:
        print(f'  Part {i+1}/{len(geo_parts)}...')
        show_ram()

    # Pair with matching transactions part
    part_name = os.path.basename(geo_p)
    txn_p = txn_by_name.get(part_name)
    if txn_p is None:
        txn_p = txn_parts[i] if i < len(txn_parts) else None
    if txn_p is None:
        continue

    # Load transactions: only transaction_id, account_id, timestamp
    txn_df = pl.read_parquet(txn_p, columns=['transaction_id','account_id','transaction_timestamp'])
    txn_df = txn_df.with_columns(
        pl.col('transaction_timestamp').str.to_datetime(format=None, strict=False).alias('ts')
    ).drop('transaction_timestamp')

    # Load geo/additional
    geo_df = pl.read_parquet(geo_p, columns=[
        'transaction_id','latitude','longitude','ip_address',
        'balance_after_transaction','part_transaction_type','transaction_sub_type'
    ])

    # ── KEY FIX: Polars join replaces the entire txn_id_map dict ──
    joined = txn_df.join(geo_df, on='transaction_id', how='inner')
    del txn_df, geo_df

    # Attach mobile-change date for post-mobile detection
    joined = joined.join(mob_pl, on='account_id', how='left')
    joined = joined.with_columns(
        ((pl.col('ts') - pl.col('last_mobile_update_date')).dt.total_days()).alias('days_since_mob')
    )

    # ── Vectorized group_by aggregations (all C-speed in Polars) ──

    # 1. Geo spread
    geo_agg = (
        joined
        .filter(pl.col('latitude').is_not_null() & pl.col('longitude').is_not_null())
        .group_by('account_id')
        .agg([
            pl.col('latitude').std().alias('lat_std'),
            pl.col('longitude').std().alias('lon_std'),
            (pl.col('latitude').max() - pl.col('latitude').min()).alias('lat_range'),
            (pl.col('longitude').max() - pl.col('longitude').min()).alias('lon_range'),
        ])
    ).to_pandas()

    # 2. Balance stats
    bal_agg = (
        joined
        .filter(pl.col('balance_after_transaction').is_not_null())
        .group_by('account_id')
        .agg([
            pl.col('balance_after_transaction').std().alias('bal_std'),
            pl.col('balance_after_transaction').min().alias('bal_min'),
            pl.col('balance_after_transaction').max().alias('bal_max'),
            (pl.col('balance_after_transaction') < 1000).sum().alias('near_zero_cnt'),
            pl.col('balance_after_transaction').count().alias('bal_cnt'),
        ])
    ).to_pandas()

    # 3. CI/BI/CLT_CASH counts
    type_agg = (
        joined
        .group_by('account_id')
        .agg([
            (pl.col('part_transaction_type') == 'CI').sum().alias('ci'),
            (pl.col('part_transaction_type') == 'BI').sum().alias('bi'),
            (pl.col('transaction_sub_type') == 'CLT_CASH').sum().alias('cash'),
        ])
    ).to_pandas()

    # 4. Post-mobile-change transaction count
    mob_agg = (
        joined
        .filter(
            pl.col('days_since_mob').is_not_null() &
            (pl.col('days_since_mob') >= 0) &
            (pl.col('days_since_mob') <= 30)
        )
        .group_by('account_id')
        .agg(pl.len().alias('post_mob'))
    ).to_pandas()

    # 5. IP aggregation (small output — OK in pandas)
    ip_df = (
        joined
        .filter(pl.col('ip_address').is_not_null())
        .select(['account_id','ip_address'])
    ).to_pandas()

    # ── Update accumulators with tiny summary rows ──
    for row in geo_agg.itertuples(index=False):
        g = GEO[row.account_id]
        g['lat_stds'].append(row.lat_std or 0.0)
        g['lon_stds'].append(row.lon_std or 0.0)
        g['lat_range'] += (row.lat_range or 0.0)
        g['lon_range'] += (row.lon_range or 0.0)

    for row in bal_agg.itertuples(index=False):
        GEO[row.account_id]['bals'].append(
            (row.bal_std or 0, row.bal_min or 0, row.bal_max or 0,
             row.near_zero_cnt or 0, row.bal_cnt or 0)
        )

    for row in type_agg.itertuples(index=False):
        g = GEO[row.account_id]
        g['ci']   += row.ci   or 0
        g['bi']   += row.bi   or 0
        g['cash'] += row.cash or 0

    for row in mob_agg.itertuples(index=False):
        GEO[row.account_id]['post_mob'] += row.post_mob or 0

    if not ip_df.empty:
        for aid, grp in ip_df.groupby('account_id'):
            ips = grp['ip_address'].tolist()
            GEO[aid]['ips'].update(ips)
            GEO[aid]['ip_list'].extend(ips)

    del joined, geo_agg, bal_agg, type_agg, mob_agg, ip_df
    gc.collect()

del mob_pl; gc.collect()

# ── PATTERN F: IP Subnet Sharing ──
print('Computing IP subnet features...')
subnet_accounts = defaultdict(set)
for acc_id, g in GEO.items():
    for ip in g['ips']:
        try:
            subnet_accounts['.'.join(ip.split('.')[:3])].add(acc_id)
        except:
            pass

# ── Build final feature DataFrame ──
print('Building geo feature records...')
geo_records = []
for acc_id, g in GEO.items():
    r = {'account_id': acc_id}

    # Geo spread (mean of per-part stds)
    if g['lat_stds']:
        r['geo_lat_std']   = float(np.mean(g['lat_stds']))
        r['geo_lon_std']   = float(np.mean(g['lon_stds']))
        r['geo_spread']    = r['geo_lat_std'] + r['geo_lon_std']
        r['geo_lat_range'] = float(g['lat_range'])
        r['geo_lon_range'] = float(g['lon_range'])
    else:
        r['geo_lat_std'] = r['geo_lon_std'] = r['geo_spread'] = 0.0
        r['geo_lat_range'] = r['geo_lon_range'] = 0.0

    # IP features
    r['unique_ip_count'] = len(g['ips'])

    max_subnet_overlap = 0
    for ip in g['ips']:
        try:
            overlap = len(subnet_accounts['.'.join(ip.split('.')[:3])]) - 1
            if overlap > max_subnet_overlap:
                max_subnet_overlap = overlap
        except:
            pass
    r['ip_subnet_sharing'] = max_subnet_overlap

    if g['ip_list']:
        top = Counter(g['ip_list']).most_common(1)[0][1]
        r['single_ip_dominance'] = top / len(g['ip_list'])
    else:
        r['single_ip_dominance'] = 0.0

    # Balance features
    if g['bals']:
        stds  = [b[0] for b in g['bals']]
        mins  = [b[1] for b in g['bals']]
        maxs  = [b[2] for b in g['bals']]
        nz    = sum(b[3] for b in g['bals'])
        cnt   = sum(b[4] for b in g['bals'])
        r['balance_volatility']       = float(np.mean(stds))
        r['min_balance_ever']         = float(min(mins))
        r['max_balance_ever']         = float(max(maxs))
        r['balance_range']            = r['max_balance_ever'] - r['min_balance_ever']
        r['near_zero_balance_count']  = nz
        r['balance_floor_ratio']      = nz / max(cnt, 1)
        r['balance_stagnation_score'] = float(np.mean(stds)) / max(r['max_balance_ever'], 1)
    else:
        for k in ['balance_volatility','min_balance_ever','max_balance_ever',
                  'balance_range','near_zero_balance_count',
                  'balance_floor_ratio','balance_stagnation_score']:
            r[k] = 0.0

    total_type = max(g['ci'] + g['bi'], 1)
    r['ci_ratio']              = g['ci'] / total_type
    r['clt_cash_count']        = g['cash']
    r['post_mobile_txn_count'] = g['post_mob']
    geo_records.append(r)

geo_features = pd.DataFrame(geo_records)
geo_features.to_parquet('geo_features.parquet', index=False)
print(f'Geo features saved: {geo_features.shape}')
print('Features: geo_spread, ip_subnet_sharing, single_ip_dominance, balance_floor_ratio, balance_stagnation_score, post_mobile_txn_count')

del GEO, subnet_accounts, geo_records, geo_features
gc.collect()
show_ram()
print('✅ STEP 2C COMPLETE — geo_features.parquet saved')


---
# 🔧 STEP 2D: Network/Graph Features (NEW — Pattern A)
### counterparty_mule_overlap, network_mule_exposure
Memory safe: only uses train labels + counterparty sets already built.

In [ ]:
print('Building counterparty network features...')
show_ram()

# Load train labels for mule account IDs
tl = pd.read_parquet('train_labels.parquet', columns=['account_id','is_mule'])
mule_account_ids = set(tl[tl['is_mule']==1]['account_id'].tolist())
del tl; clear_memory()

print(f'Known mule accounts: {len(mule_account_ids):,}')

# Load transaction features to get counterparty counts
tf = pd.read_parquet('transaction_features.parquet',
    columns=['account_id','unique_credit_counterparties','unique_debit_counterparties'])

# Scan transactions to find counterparty overlap with mules
# Process chunk by chunk — only need account_id and counterparty_id
print('Scanning transactions for mule counterparty overlap...')

# acc -> set of counterparties that are mules
MULE_CP_CREDIT = defaultdict(int)   # count of mule counterparties in credits
MULE_CP_DEBIT  = defaultdict(int)   # count of mule counterparties in debits
TOTAL_CP       = defaultdict(set)   # all counterparties seen

for i, p in enumerate(sorted(glob('transactions/batch-*/part_*.parquet'))):
    if i % 30 == 0: print(f'  Part {i+1}...')
    df = pl.read_parquet(p, columns=['account_id','txn_type','counterparty_id'])
    df = df.filter(pl.col('counterparty_id').is_not_null()).to_pandas()

    for row in df.itertuples(index=False):
        cp = row.counterparty_id
        aid = row.account_id
        TOTAL_CP[aid].add(cp)
        if cp in mule_account_ids:
            if row.txn_type == 'C':
                MULE_CP_CREDIT[aid] += 1
            else:
                MULE_CP_DEBIT[aid] += 1

    del df; clear_memory()

# Build network feature records
net_records = []
all_acc_ids = set(TOTAL_CP.keys()) | set(MULE_CP_CREDIT.keys()) | set(MULE_CP_DEBIT.keys())
for acc_id in all_acc_ids:
    total_cp = len(TOTAL_CP.get(acc_id, set()))
    mule_cr  = MULE_CP_CREDIT.get(acc_id, 0)
    mule_dr  = MULE_CP_DEBIT.get(acc_id, 0)
    total_mule_cp = mule_cr + mule_dr

    net_records.append({
        'account_id': acc_id,
        'mule_counterparty_credit_count': mule_cr,
        'mule_counterparty_debit_count':  mule_dr,
        'mule_counterparty_total':        total_mule_cp,
        'mule_counterparty_ratio':        total_mule_cp / max(total_cp, 1),
        'has_mule_counterparty':          int(total_mule_cp > 0),
    })

net_features = pd.DataFrame(net_records)
net_features.to_parquet('network_features.parquet', index=False)
print(f'Network features: {net_features.shape}')
print(f'Accounts with mule counterparty: {net_features["has_mule_counterparty"].sum():,}')

del MULE_CP_CREDIT, MULE_CP_DEBIT, TOTAL_CP, net_records
del net_features, tf, mule_account_ids
clear_memory()
show_ram()
print('✅ STEP 2D COMPLETE — network_features.parquet saved')

---
# 🧹 STEP 3: Red-Herring Detection
### NEW: Cluster-aware Isolation Forest (Pattern per peer group)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans
import lightgbm as lgb

show_ram()

# Load all feature files
sf  = pd.read_parquet('static_features.parquet')
tf  = pd.read_parquet('transaction_features.parquet')
gf  = pd.read_parquet('geo_features.parquet')
nf  = pd.read_parquet('network_features.parquet')
tl  = pd.read_parquet('train_labels.parquet', columns=['account_id','is_mule'])

features = sf.merge(tf,on='account_id',how='left')              .merge(gf,on='account_id',how='left')              .merge(nf,on='account_id',how='left')
del sf, tf, gf, nf; clear_memory()

train_data = features[features['account_id'].isin(tl['account_id'])].merge(tl,on='account_id',how='left')
del features, tl; clear_memory()
show_ram()

feat_cols = [c for c in train_data.columns
             if c not in ['account_id','is_mule']
             and train_data[c].dtype in [np.float64,np.int64,np.float32,np.int32,float,int]]
X = train_data[feat_cols].fillna(0).astype(np.float32)
y = train_data['is_mule'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)
del X; clear_memory()

# ── GLOBAL Isolation Forest ──
print('Running global Isolation Forest...')
iso_global = IsolationForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
iso_global.fit(X_scaled)
global_anomaly = iso_global.decision_function(X_scaled)
del iso_global; clear_memory()

# ── CLUSTER-AWARE Isolation Forest (NEW) ──
# Cluster by product family + scheme + branch type (already encoded)
print('Running cluster-aware Isolation Forest...')
cluster_cols_idx = [i for i, c in enumerate(feat_cols)
                    if c in ['product_family_enc','branch_type_enc','branch_mule_rate',
                             'scheme_PMJDY','scheme_REGULAR','rural_branch_enc']]
if len(cluster_cols_idx) > 1:
    X_cluster = X_scaled[:, cluster_cols_idx]
    km = KMeans(n_clusters=8, random_state=42, n_init=5)
    cluster_labels = km.fit_predict(X_cluster)
    del km, X_cluster; clear_memory()

    cluster_anomaly = np.zeros(len(X_scaled))
    for cl in range(8):
        mask = cluster_labels == cl
        if mask.sum() < 50: continue
        iso_cl = IsolationForest(n_estimators=80, contamination=0.05,
                                  random_state=42, n_jobs=-1)
        iso_cl.fit(X_scaled[mask])
        cluster_anomaly[mask] = iso_cl.decision_function(X_scaled[mask])
        del iso_cl; clear_memory()

    # Combine: take min (most anomalous) of global and cluster scores
    combined_anomaly = np.minimum(global_anomaly, cluster_anomaly)
    print(f'Cluster distribution: {dict(zip(*np.unique(cluster_labels, return_counts=True)))}')
else:
    combined_anomaly = global_anomaly

train_data['anomaly_score'] = combined_anomaly
del global_anomaly; clear_memory()

# ── OOF LightGBM for label confidence ──
print('Running OOF LightGBM for red-herring detection...')
oof_probs = np.zeros(len(y))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold,(tri,vali) in enumerate(skf.split(X_scaled,y)):
    print(f'  Fold {fold+1}/5...')
    m = lgb.LGBMClassifier(n_estimators=200,learning_rate=0.05,num_leaves=63,
                             random_state=42,n_jobs=-1,class_weight='balanced')
    m.fit(X_scaled[tri],y[tri],eval_set=[(X_scaled[vali],y[vali])],
          callbacks=[lgb.early_stopping(20,verbose=False),lgb.log_evaluation(False)])
    oof_probs[vali] = m.predict_proba(X_scaled[vali])[:,1]
    del m; clear_memory()

print(f'OOF AUC: {roc_auc_score(y, oof_probs):.4f}')

train_data['oof_prob']          = oof_probs
train_data['label_confidence']  = 1.0
rh_mask = (train_data['is_mule']==1) & (train_data['oof_prob'] < 0.2)
train_data.loc[rh_mask,'label_confidence'] = 0.3
train_data['soft_label']            = 0.7*train_data['is_mule'] + 0.3*train_data['oof_prob']
train_data['suspected_red_herring'] = rh_mask.astype(int)
print(f'Suspected red herrings: {rh_mask.sum()}')

cl = train_data[['account_id','is_mule','soft_label','label_confidence',
                  'anomaly_score','oof_prob','suspected_red_herring']].copy()
cl.to_parquet('clean_labels.parquet', index=False)

del train_data, X_scaled, cl; clear_memory()
show_ram()
print('✅ STEP 3 COMPLETE — clean_labels.parquet saved')

---
# 🤖 STEP 4: Model Training
### Series + Parallel Hybrid Architecture
### Stage 1: Screening → Stage 2: Deep model on high-risk only → Stage 3: Meta-learner
### NEW: Autoencoder anomaly score added as extra feature

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb

show_ram()

# Load everything
sf = pd.read_parquet('static_features.parquet')
tf = pd.read_parquet('transaction_features.parquet')
gf = pd.read_parquet('geo_features.parquet')
nf = pd.read_parquet('network_features.parquet')
cl = pd.read_parquet('clean_labels.parquet')
ta = pd.read_parquet('test_accounts.parquet', columns=['account_id'])

all_feat = sf.merge(tf,on='account_id',how='left')              .merge(gf,on='account_id',how='left')              .merge(nf,on='account_id',how='left')
del sf, tf, gf, nf; clear_memory()

train_feat = all_feat[all_feat['account_id'].isin(cl['account_id'])].merge(cl,on='account_id',how='left')
test_feat  = all_feat[all_feat['account_id'].isin(ta['account_id'])].copy()
del all_feat; clear_memory()

EXCL = ['account_id','is_mule','soft_label','label_confidence',
        'anomaly_score','oof_prob','suspected_red_herring']
feat_cols = [c for c in train_feat.columns
             if c not in EXCL
             and train_feat[c].dtype in [np.float64,np.int64,np.float32,np.int32,float,int]]

not_rh = train_feat['suspected_red_herring'] == 0
train_clean = train_feat[not_rh].copy()

# ══════════════════════════════════════════════════════════
# TEMPORAL VALIDATION (Phase 2 requirement)
# Train on older accounts → validate on recent accounts
# Simulates real drift: model trained on past, tested on future
# ══════════════════════════════════════════════════════════
print('Setting up TEMPORAL validation split...')

# Use account_age_days as proxy for time
# Older accounts (higher age) = earlier opened = training
# Newer accounts (lower age)  = recently opened = validation
if 'account_age_days' in train_clean.columns:
    age_median = train_clean['account_age_days'].median()
    temporal_train_mask = train_clean['account_age_days'] >= age_median
    temporal_val_mask   = train_clean['account_age_days'] <  age_median
    print(f'Temporal train (older accounts): {temporal_train_mask.sum():,}')
    print(f'Temporal val   (newer accounts): {temporal_val_mask.sum():,}')
    print(f'Mule rate in train: {train_clean.loc[temporal_train_mask,"is_mule"].mean():.3f}')
    print(f'Mule rate in val:   {train_clean.loc[temporal_val_mask,"is_mule"].mean():.3f}')
else:
    # Fallback to stratified if age not available
    temporal_train_mask = pd.Series([True]*len(train_clean))
    temporal_val_mask   = pd.Series([False]*len(train_clean))
    print('account_age_days not found — using full dataset for CV')

X_temporal_tr = train_clean.loc[temporal_train_mask, feat_cols].fillna(0).astype(np.float32).values
y_temporal_tr = train_clean.loc[temporal_train_mask, 'is_mule'].values
X_temporal_val= train_clean.loc[temporal_val_mask,   feat_cols].fillna(0).astype(np.float32).values
y_temporal_val= train_clean.loc[temporal_val_mask,   'is_mule'].values

# Full arrays for cross-validation
X_tr = train_clean[feat_cols].fillna(0).astype(np.float32).values
y_tr = train_clean['is_mule'].values
w_tr = train_clean['label_confidence'].values
X_te = test_feat[feat_cols].fillna(0).astype(np.float32).values

del train_feat, test_feat, train_clean; clear_memory()

spw = (y_tr==0).sum() / (y_tr==1).sum()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'\nTrain: {X_tr.shape} | Mule rate: {y_tr.mean():.3f} | Class weight: {spw:.1f}')

# ── IMBALANCE REPORT ──
print('\n' + '='*50)
print('IMBALANCE ANALYSIS')
print('='*50)
mule_count  = (y_tr==1).sum()
legit_count = (y_tr==0).sum()
print(f'Mule accounts:  {mule_count:,}  ({mule_count/len(y_tr)*100:.1f}%)')
print(f'Legit accounts: {legit_count:,} ({legit_count/len(y_tr)*100:.1f}%)')
print(f'Imbalance ratio: 1:{int(legit_count/mule_count)}')
print(f'Strategy: scale_pos_weight={spw:.1f} + sample_weight from label_confidence')
print(f'Red herrings removed: {(cl["suspected_red_herring"]==1).sum()}')
show_ram()


In [ ]:
# ── AUTOENCODER ANOMALY SCORE (NEW) ──
# Train on LEGIT accounts only. High reconstruction error = suspicious.
print('Training Autoencoder on legitimate accounts...')

try:
    import tensorflow as tf
    from tensorflow import keras

    legit_mask = y_tr == 0
    X_legit = X_tr[legit_mask]

    # Normalize
    ae_scaler = StandardScaler()
    X_legit_sc = ae_scaler.fit_transform(X_legit).astype(np.float32)
    X_tr_sc    = ae_scaler.transform(X_tr).astype(np.float32)
    X_te_sc    = ae_scaler.transform(X_te).astype(np.float32)

    # Build autoencoder
    input_dim = X_tr_sc.shape[1]
    enc_dim   = max(input_dim // 4, 8)

    inp = keras.Input(shape=(input_dim,))
    enc = keras.layers.Dense(enc_dim*2, activation='relu')(inp)
    enc = keras.layers.Dense(enc_dim,   activation='relu')(enc)
    dec = keras.layers.Dense(enc_dim*2, activation='relu')(enc)
    out = keras.layers.Dense(input_dim, activation='linear')(dec)
    ae  = keras.Model(inp, out)
    ae.compile(optimizer='adam', loss='mse')

    ae.fit(X_legit_sc, X_legit_sc,
           epochs=30, batch_size=512,
           validation_split=0.1, verbose=0)

    # Reconstruction error = anomaly score
    tr_recon = ae.predict(X_tr_sc, verbose=0)
    te_recon = ae.predict(X_te_sc, verbose=0)
    ae_score_tr = np.mean((X_tr_sc - tr_recon)**2, axis=1)
    ae_score_te = np.mean((X_te_sc - te_recon)**2, axis=1)

    print(f'Autoencoder trained. Avg recon error — legit: {ae_score_tr[legit_mask].mean():.4f} | mule: {ae_score_tr[~legit_mask].mean():.4f}')
    use_autoencoder = True

    del ae, X_legit, X_legit_sc, X_tr_sc, X_te_sc, tr_recon, te_recon
    clear_memory()

except Exception as e:
    print(f'Autoencoder skipped ({e}). Adding zero score.')
    ae_score_tr = np.zeros(len(X_tr))
    ae_score_te = np.zeros(len(X_te))
    use_autoencoder = False

# Append autoencoder score as an extra feature column
X_tr = np.hstack([X_tr, ae_score_tr.reshape(-1,1).astype(np.float32)])
X_te = np.hstack([X_te, ae_score_te.reshape(-1,1).astype(np.float32)])
print(f'Feature matrix after AE score: {X_tr.shape}')
show_ram()

In [ ]:
# ── STAGE 1: SCREENING with static features only ──
# Use first N cols (static features) to get rough risk score fast
print('\n=== STAGE 1: Screening ===')

# Static features are approximately the first 40 columns
static_col_count = min(40, X_tr.shape[1])
X_tr_static = X_tr[:, :static_col_count]
X_te_static = X_te[:, :static_col_count]

stage1_oof  = np.zeros(len(X_tr))
stage1_test = np.zeros(len(X_te))

for fold,(tri,vali) in enumerate(skf.split(X_tr_static, y_tr)):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                             num_leaves=63, random_state=42,
                             n_jobs=-1, class_weight='balanced', verbose=-1)
    m.fit(X_tr_static[tri], y_tr[tri],
          eval_set=[(X_tr_static[vali], y_tr[vali])],
          callbacks=[lgb.early_stopping(20,verbose=False), lgb.log_evaluation(False)])
    stage1_oof[vali]  = m.predict_proba(X_tr_static[vali])[:,1]
    stage1_test      += m.predict_proba(X_te_static)[:,1] / 5
    del m; clear_memory()

stage1_auc = roc_auc_score(y_tr, stage1_oof)
print(f'Stage 1 AUC: {stage1_auc:.4f}')

# Identify HIGH RISK accounts (top 30% by stage1 score)
HIGH_RISK_THRESH = np.percentile(stage1_oof, 70)
high_risk_mask_tr = stage1_oof >= HIGH_RISK_THRESH
high_risk_mask_te = stage1_test >= np.percentile(stage1_test, 70)
print(f'High-risk training accounts: {high_risk_mask_tr.sum():,} / {len(X_tr):,}')

del X_tr_static, X_te_static; clear_memory()
show_ram()

In [ ]:
# ── STAGE 2A: LightGBM (full features) ──
print('\n=== STAGE 2: LightGBM (full features) ===')
lgb_oof  = stage1_oof.copy()   # seed with stage1 for low-risk accounts
lgb_test = stage1_test.copy()

for fold,(tri,vali) in enumerate(skf.split(X_tr, y_tr)):
    print(f'  LGB Fold {fold+1}/5...')
    m = lgb.LGBMClassifier(
        n_estimators=800, learning_rate=0.03, num_leaves=127,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=42, n_jobs=-1, class_weight='balanced')
    m.fit(X_tr[tri], y_tr[tri], sample_weight=w_tr[tri],
          eval_set=[(X_tr[vali], y_tr[vali])],
          callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(200)])
    lgb_oof[vali]  = m.predict_proba(X_tr[vali])[:,1]
    lgb_test      += m.predict_proba(X_te)[:,1] / 5
    del m; clear_memory()

lgb_auc = roc_auc_score(y_tr, lgb_oof)
print(f'LightGBM AUC: {lgb_auc:.4f}')
show_ram()

In [ ]:
# ── STAGE 2B: XGBoost ──
print('\n=== STAGE 2: XGBoost ===')
xgb_oof  = np.zeros(len(X_tr))
xgb_test = np.zeros(len(X_te))

for fold,(tri,vali) in enumerate(skf.split(X_tr, y_tr)):
    print(f'  XGB Fold {fold+1}/5...')
    m = xgb.XGBClassifier(
        n_estimators=800, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, random_state=42,
        n_jobs=-1, eval_metric='auc',
        early_stopping_rounds=50, verbosity=0)
    m.fit(X_tr[tri], y_tr[tri], eval_set=[(X_tr[vali],y_tr[vali])], verbose=False)
    xgb_oof[vali]  = m.predict_proba(X_tr[vali])[:,1]
    xgb_test      += m.predict_proba(X_te)[:,1] / 5
    del m; clear_memory()

xgb_auc = roc_auc_score(y_tr, xgb_oof)
print(f'XGBoost AUC: {xgb_auc:.4f}')
show_ram()

In [ ]:
# ── OPTIONAL: CatBoost (run only if RAM > 5GB free) ──
import psutil
free_gb = psutil.virtual_memory().available / 1e9
print(f'Free RAM: {free_gb:.1f}GB')

if free_gb > 5.0:
    print('\n=== STAGE 2: CatBoost (RAM sufficient) ===')
    try:
        from catboost import CatBoostClassifier
        cb_oof  = np.zeros(len(X_tr))
        cb_test = np.zeros(len(X_te))
        for fold,(tri,vali) in enumerate(skf.split(X_tr,y_tr)):
            print(f'  CB Fold {fold+1}/5...')
            m = CatBoostClassifier(iterations=600,learning_rate=0.03,depth=6,
                random_seed=42,eval_metric='AUC',early_stopping_rounds=50,
                verbose=False,class_weights=[1,spw])
            m.fit(X_tr[tri],y_tr[tri],eval_set=(X_tr[vali],y_tr[vali]))
            cb_oof[vali]  = m.predict_proba(X_tr[vali])[:,1]
            cb_test      += m.predict_proba(X_te)[:,1] / 5
            del m; clear_memory()
        cb_auc = roc_auc_score(y_tr, cb_oof)
        print(f'CatBoost AUC: {cb_auc:.4f}')
        use_catboost = True
    except Exception as e:
        print(f'CatBoost failed: {e}')
        cb_oof = lgb_oof.copy(); cb_test = lgb_test.copy(); cb_auc = lgb_auc
        use_catboost = False
else:
    print('Skipping CatBoost — not enough RAM')
    cb_oof = lgb_oof.copy(); cb_test = lgb_test.copy(); cb_auc = lgb_auc
    use_catboost = False

show_ram()

In [ ]:
# ── STAGE 3: META-LEARNER (stacking) ──
print('\n=== STAGE 3: Meta-Learner ===')

oof_stack  = np.column_stack([stage1_oof, lgb_oof, xgb_oof, cb_oof, ae_score_tr])
test_stack = np.column_stack([stage1_test, lgb_test, xgb_test, cb_test, ae_score_te])

meta = LogisticRegression(C=1.0, random_state=42)
meta.fit(oof_stack, y_tr)
meta_oof  = meta.predict_proba(oof_stack)[:,1]
meta_test = meta.predict_proba(test_stack)[:,1]
meta_auc  = roc_auc_score(y_tr, meta_oof)
print(f'Meta-learner AUC: {meta_auc:.4f}')
print(f'Model weights: Stage1={meta.coef_[0][0]:.3f} LGB={meta.coef_[0][1]:.3f} XGB={meta.coef_[0][2]:.3f} CB={meta.coef_[0][3]:.3f} AE={meta.coef_[0][4]:.3f}')

# Best F1 threshold
best_f1, best_thresh = 0, 0.5
for t in np.arange(0.1, 0.9, 0.02):
    f1 = f1_score(y_tr, (meta_oof>=t).astype(int), zero_division=0)
    if f1 > best_f1: best_f1=f1; best_thresh=t
print(f'Best threshold: {best_thresh:.2f} → F1: {best_f1:.4f}')

preds = ta.copy()
preds['is_mule_prob']   = meta_test
preds['is_mule_binary'] = (meta_test >= best_thresh).astype(int)
preds.to_parquet('final_model_predictions.parquet', index=False)

del X_tr, X_te, lgb_oof, lgb_test, xgb_oof, xgb_test
del cb_oof, cb_test, stage1_oof, stage1_test
del oof_stack, test_stack, meta_oof, meta_test, ae_score_tr, ae_score_te
clear_memory()
show_ram()
print(f'✅ STEP 4 COMPLETE — Predicted mules: {preds["is_mule_binary"].sum()}')
# ── PROBABILITY CALIBRATION ──
# Ensures probabilities are well-calibrated for AUC scoring
# Critical: AUC-ROC is the primary ranking metric
from sklearn.calibration import CalibratedClassifierCV
print('\nCalibrating probabilities...')

# Isotonic calibration on OOF stacked predictions
from sklearn.isotonic import IsotonicRegression
iso_cal = IsotonicRegression(out_of_bounds='clip')
iso_cal.fit(meta_oof, y_tr)
meta_oof_cal  = iso_cal.predict(meta_oof)
meta_test_cal = iso_cal.predict(meta_test)

auc_before = roc_auc_score(y_tr, meta_oof)
auc_after  = roc_auc_score(y_tr, meta_oof_cal)
print(f'AUC before calibration: {auc_before:.4f}')
print(f'AUC after calibration:  {auc_after:.4f}')

# Use calibrated if it helps, else keep original
if auc_after >= auc_before - 0.001:
    final_test_probs = meta_test_cal
    final_oof_probs  = meta_oof_cal
    print('✅ Using calibrated probabilities')
else:
    final_test_probs = meta_test
    final_oof_probs  = meta_oof
    print('Using original probabilities (calibration did not help)')

# Update predictions with calibrated probs
preds['is_mule_prob']   = final_test_probs
preds['is_mule_binary'] = (final_test_probs >= best_thresh).astype(int)
preds.to_parquet('final_model_predictions.parquet', index=False)
print(f'Updated predictions saved. Mules: {preds["is_mule_binary"].sum()}')


---
# 📉 STEP 4B: Imbalance & Drift Evaluation (Phase 2 Requirement)
### Tests model robustness across time periods — older vs newer accounts

In [ ]:
# ══════════════════════════════════════════════════════════
# DRIFT VALIDATION — evaluate on temporal holdout
# ══════════════════════════════════════════════════════════
print('='*55)
print('TEMPORAL DRIFT VALIDATION')
print('='*55)

if len(X_temporal_val) > 0 and len(np.unique(y_temporal_val)) > 1:
    # Train quick LGB on older accounts only
    lgb_temporal = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        random_state=42, n_jobs=-1, class_weight='balanced', verbose=-1)
    lgb_temporal.fit(X_temporal_tr, y_temporal_tr)

    # Evaluate on newer accounts
    val_probs = lgb_temporal.predict_proba(X_temporal_val)[:,1]
    val_preds = (val_probs >= 0.5).astype(int)

    temporal_auc = roc_auc_score(y_temporal_val, val_probs)
    temporal_f1  = f1_score(y_temporal_val, val_preds, zero_division=0)
    temporal_pr  = precision_score(y_temporal_val, val_preds, zero_division=0)
    temporal_rec = recall_score(y_temporal_val, val_preds, zero_division=0)

    print(f'Model trained on OLDER accounts, tested on NEWER:')
    print(f'  AUC:       {temporal_auc:.4f}')
    print(f'  F1:        {temporal_f1:.4f}')
    print(f'  Precision: {temporal_pr:.4f}')
    print(f'  Recall:    {temporal_rec:.4f}')

    # Compare with overall CV AUC
    overall_auc = roc_auc_score(y_tr, meta_oof) if 'meta_oof' in dir() else lgb_auc
    drift_gap = overall_auc - temporal_auc
    print(f'\nDrift gap (CV AUC - Temporal AUC): {drift_gap:.4f}')
    if drift_gap < 0.02:
        print('✅ LOW DRIFT — Model generalizes well across time')
    elif drift_gap < 0.05:
        print('⚠️  MODERATE DRIFT — Some temporal degradation')
    else:
        print('❌ HIGH DRIFT — Model struggles on recent accounts')

    del lgb_temporal; clear_memory()
else:
    print('Temporal validation not available — skipping')

# ── THRESHOLD ANALYSIS (imbalance handling) ──
print('\n' + '='*55)
print('PRECISION-RECALL THRESHOLD ANALYSIS')
print('='*55)
print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Predicted Mules":>16}')
print('-'*60)

meta_scores = meta_oof if 'meta_oof' in dir() else lgb_oof
for thresh in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    prd = (meta_scores >= thresh).astype(int)
    pr  = precision_score(y_tr, prd, zero_division=0)
    rc  = recall_score(y_tr, prd, zero_division=0)
    f1  = f1_score(y_tr, prd, zero_division=0)
    cnt = prd.sum()
    marker = ' ← best F1' if abs(thresh - best_thresh) < 0.01 else ''
    print(f'{thresh:>10.2f} {pr:>10.4f} {rc:>10.4f} {f1:>10.4f} {cnt:>16,}{marker}')

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# PR Curve
prec, rec, _ = precision_recall_curve(y_tr, meta_scores)
axes[0].plot(rec, prec, color='navy', lw=2)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].fill_between(rec, prec, alpha=0.1, color='navy')

# ROC Curve
fpr, tpr, _ = roc_curve(y_tr, meta_scores)
auc_val = roc_auc_score(y_tr, meta_scores)
axes[1].plot(fpr, tpr, color='crimson', lw=2, label=f'AUC={auc_val:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()

# Score distribution
axes[2].hist(meta_scores[y_tr==0], bins=50, alpha=0.6, label='Legit', color='steelblue', density=True)
axes[2].hist(meta_scores[y_tr==1], bins=50, alpha=0.6, label='Mule',  color='crimson',   density=True)
axes[2].axvline(best_thresh, color='black', linestyle='--', label=f'Threshold={best_thresh:.2f}')
axes[2].set_title('Score Distribution')
axes[2].set_xlabel('Predicted Probability')
axes[2].legend()

plt.tight_layout()
plt.savefig('plot_imbalance_drift.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close()
print('\n✅ Plot saved: plot_imbalance_drift.png')


---
# 🔍 STEP 4C: Explainability (XAI) — Phase 2 Requirement
### SHAP values for transparency and regulator-friendly interpretation

In [ ]:
# ══════════════════════════════════════════════════════════
# SHAP EXPLAINABILITY
# Required for Phase 2: Transparency and interpretability
# ══════════════════════════════════════════════════════════
import shap
import matplotlib.pyplot as plt

print('Computing SHAP values...')
print('(Using last LightGBM fold model)')
show_ram()

# Reload LGB model on full training data for SHAP
lgb_shap = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    random_state=42, n_jobs=-1, class_weight='balanced', verbose=-1)
lgb_shap.fit(X_tr, y_tr, sample_weight=w_tr)

# Use a sample for SHAP (full dataset too slow on free Colab)
# 2000 samples is enough for reliable SHAP values
sample_size = min(2000, len(X_tr))
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_tr), sample_size, replace=False)
X_sample = X_tr[sample_idx]
y_sample = y_tr[sample_idx]

print(f'Computing SHAP on {sample_size} samples...')
explainer   = shap.TreeExplainer(lgb_shap)
shap_values = explainer.shap_values(X_sample)

# shap_values is list [class0, class1] for binary classification
if isinstance(shap_values, list):
    sv = shap_values[1]   # class 1 = mule
else:
    sv = shap_values

print(f'SHAP values shape: {sv.shape}')

# ── PLOT 1: Global Feature Importance (SHAP Summary) ──
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_sample, feature_names=feat_cols,
                  max_display=25, show=False, plot_type='bar')
plt.title('SHAP Feature Importance — Top 25 Features')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Saved: shap_importance.png')
clear_memory()


In [ ]:
import shap
# ── PLOT 2: SHAP Beeswarm (impact direction) ──
plt.figure(figsize=(12, 9))
shap.summary_plot(sv, X_sample, feature_names=feat_cols,
                  max_display=20, show=False)
plt.title('SHAP Beeswarm — Feature Impact on Mule Prediction')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Saved: shap_beeswarm.png')

# ── TOP FEATURES REPORT ──
print('\n' + '='*55)
print('TOP 15 MOST IMPORTANT FEATURES (SHAP)')
print('='*55)
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feat_cols).sort_values(ascending=False)
for i, (feat, val) in enumerate(shap_importance.head(15).items()):
    print(f'  {i+1:2d}. {feat:<45} {val:.4f}')

shap_importance.to_csv('shap_feature_importance.csv')
print('\n✅ Saved: shap_feature_importance.csv')
clear_memory()


In [ ]:
import shap
# ── PLOT 3: Explain individual mule predictions ──
# Pick top 5 most confidently predicted mules and explain WHY
print('\n' + '='*55)
print('INDIVIDUAL MULE EXPLANATIONS (Top 5 predicted mules)')
print('='*55)

mule_sample_idx = np.where(y_sample == 1)[0]
if len(mule_sample_idx) > 0:
    top_mules = mule_sample_idx[:min(5, len(mule_sample_idx))]

    for rank, idx in enumerate(top_mules):
        print(f'\n--- Mule #{rank+1} (sample index {idx}) ---')
        top_features_idx = np.argsort(np.abs(sv[idx]))[::-1][:5]
        for feat_idx in top_features_idx:
            fname = feat_cols[feat_idx]
            fval  = X_sample[idx, feat_idx]
            sval  = sv[idx, feat_idx]
            direction = '▲ pushes toward MULE' if sval > 0 else '▼ pushes toward LEGIT'
            print(f'  {fname:<40} value={fval:>10.2f}  SHAP={sval:>+.4f}  {direction}')
else:
    print('No mule accounts in sample — increase sample_size or check labels')

# ── PLOT 4: SHAP Dependence for top 3 features ──
top3 = shap_importance.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, feat in enumerate(top3):
    feat_idx = feat_cols.index(feat)
    axes[i].scatter(X_sample[:, feat_idx], sv[:, feat_idx],
                    c=y_sample, cmap='RdBu_r', alpha=0.4, s=10)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('SHAP value')
    axes[i].set_title(f'SHAP Dependence: {feat}')
    axes[i].axhline(0, color='black', lw=0.5)
plt.tight_layout()
plt.savefig('shap_dependence.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close()
print('\n✅ Saved: shap_dependence.png')

del lgb_shap, explainer, shap_values, sv, X_sample
clear_memory()
show_ram()
print('\n✅ STEP 4C COMPLETE — XAI done')
print('\nFiles saved:')
print('  shap_importance.png     — global feature importance bar chart')
print('  shap_beeswarm.png       — feature impact direction')
print('  shap_dependence.png     — top 3 feature dependence plots')
print('  shap_feature_importance.csv — ranked feature importances')


In [ ]:
import shap
# ── GENERATE MULE EXPLANATION REPORT (per account) ──
# For each predicted mule in TEST set, save top 3 reasons
# This makes your submission regulator-friendly
print('Generating per-account explanation report...')

# Reload model and compute SHAP on test set
lgb_explain = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    random_state=42, n_jobs=-1, class_weight='balanced', verbose=-1)
lgb_explain.fit(X_tr, y_tr, sample_weight=w_tr)

explainer_te = shap.TreeExplainer(lgb_explain)
shap_te      = explainer_te.shap_values(X_te)
sv_te = shap_te[1] if isinstance(shap_te, list) else shap_te

preds_loaded = pd.read_parquet('final_model_predictions.parquet')
mule_mask_te = preds_loaded['is_mule_binary'] == 1
mule_indices = np.where(mule_mask_te.values)[0]

print(f'Generating explanations for {len(mule_indices):,} predicted mules...')

explanation_rows = []
for idx in mule_indices:
    acc_id = preds_loaded.iloc[idx]['account_id']
    top3_feat_idx = np.argsort(np.abs(sv_te[idx]))[::-1][:3]
    row = {'account_id': acc_id}
    for rank, fidx in enumerate(top3_feat_idx):
        row[f'reason_{rank+1}_feature'] = feat_cols[fidx]
        row[f'reason_{rank+1}_value']   = round(float(X_te[idx, fidx]), 4)
        row[f'reason_{rank+1}_shap']    = round(float(sv_te[idx, fidx]), 4)
    explanation_rows.append(row)

explanation_df = pd.DataFrame(explanation_rows)
explanation_df.to_csv('mule_explanations.csv', index=False)
print(f'✅ Saved: mule_explanations.csv ({len(explanation_df):,} rows)')
print('\nSample explanations:')
print(explanation_df.head(5).to_string())

del lgb_explain, explainer_te, shap_te, sv_te
clear_memory()
show_ram()
print('\n✅ ALL XAI COMPLETE')


In [ ]:
# feat_cols carried from Step 4 training cell
# If running this cell standalone, reload features to get feat_cols
if 'feat_cols' not in dir():
    sf = pd.read_parquet('static_features.parquet')
    tf = pd.read_parquet('transaction_features.parquet')
    gf = pd.read_parquet('geo_features.parquet')
    nf = pd.read_parquet('network_features.parquet')
    cl = pd.read_parquet('clean_labels.parquet')
    ta_tmp = pd.read_parquet('test_accounts.parquet', columns=['account_id'])
    tmp = sf.merge(tf,on='account_id',how='left').merge(gf,on='account_id',how='left').merge(nf,on='account_id',how='left')
    tr_tmp = tmp[tmp['account_id'].isin(cl['account_id'])].merge(cl,on='account_id',how='left')
    EXCL = ['account_id','is_mule','soft_label','label_confidence','anomaly_score','oof_prob','suspected_red_herring']
    feat_cols = [c for c in tr_tmp.columns if c not in EXCL and tr_tmp[c].dtype in [float,int,'float32','int32','float64','int64']]
    not_rh = tr_tmp['suspected_red_herring']==0
    X_tr = tr_tmp.loc[not_rh,feat_cols].fillna(0).astype('float32').values
    y_tr = tr_tmp.loc[not_rh,'is_mule'].values
    w_tr = tr_tmp.loc[not_rh,'label_confidence'].values
    X_te = tmp[tmp['account_id'].isin(ta_tmp['account_id'])][feat_cols].fillna(0).astype('float32').values
    del sf,tf,gf,nf,cl,ta_tmp,tmp,tr_tmp; clear_memory()
    print(f'Reloaded feat_cols: {len(feat_cols)} features')

---
# ⏱️ STEP 5 + 6: Temporal Windows + Final Submission

In [ ]:
preds    = pd.read_parquet('final_model_predictions.parquet')
mule_accs = set(preds[preds['is_mule_binary']==1]['account_id'])
print(f'Finding windows for {len(mule_accs)} mule accounts...')

MH = defaultdict(list)
for i,p in enumerate(sorted(glob('transactions/batch-*/part_*.parquet'))):
    if i%30==0: print(f'  Part {i+1}...')
    df = pl.read_parquet(p, columns=['account_id','transaction_timestamp','amount','txn_type'])
    df = df.filter(pl.col('account_id').is_in(list(mule_accs)))
    if df.is_empty(): del df; continue
    df = df.with_columns(pl.col('transaction_timestamp').str.to_datetime(format=None, strict=False).alias('transaction_timestamp'))
    for row in df.iter_rows(named=True):
        MH[row['account_id']].append((row['transaction_timestamp'], abs(row['amount']), row['txn_type']))
    del df; clear_memory()

def detect_window(txns):
    if not txns: return None, None
    txns = sorted(txns, key=lambda x: x[0])
    best_score, best_s, best_e = -1, txns[0][0], txns[-1][0]
    for i in range(len(txns)):
        ws = txns[i][0]
        try: we = ws + pd.Timedelta(days=30)
        except: continue
        wt = [t for t in txns if ws <= t[0] <= we]
        if len(wt) < 3: continue
        score = (len(wt)*0.5
                 + sum(2   for t in wt if t[1]>50000)
                 + sum(3   for t in wt if 40000<=t[1]<50000)
                 + sum(1.5 for t in wt if t[1] in [1000,5000,10000,50000]))
        if score > best_score:
            best_score = score
            best_s = ws
            best_e = max(t[0] for t in wt)
    try:
        return best_s - pd.Timedelta(days=3), best_e + pd.Timedelta(days=3)
    except:
        return best_s, best_e

windows = []
for acc in mule_accs:
    s, e = detect_window(MH.get(acc, []))
    # If no window found, use full history span as fallback
    # Better to have a rough window than none (IoU > 0 beats IoU = 0)
    if s is None and MH.get(acc):
        txn_list = sorted(MH[acc], key=lambda x: x[0])
        s = txn_list[0][0]
        e = txn_list[-1][0]
    windows.append({'account_id':acc,'suspicious_start':s,'suspicious_end':e})
tw = pd.DataFrame(windows)
del MH, windows; clear_memory()
show_ram()
print('✅ Temporal windows done')

In [ ]:
# ══════════════════════════════════════════════════════════
# ⚡ QUICK SUBMISSION VALIDATOR
# Run this right before uploading to catch format errors
# ══════════════════════════════════════════════════════════
import pandas as pd

sub_check = pd.read_csv('submission.csv')
print('=== SUBMISSION VALIDATION ===')
print(f'Rows:    {len(sub_check):,}  (expected 64,000)')
print(f'Columns: {sub_check.columns.tolist()}')
print(f'\nScore range:')
print(f'  Min: {sub_check["is_mule"].min():.6f}')
print(f'  Max: {sub_check["is_mule"].max():.6f}')
print(f'  Mean: {sub_check["is_mule"].mean():.6f}')
print(f'\nPredicted mules (>0.5):  {(sub_check["is_mule"]>0.5).sum():,}')
print(f'Predicted legit (<0.5):  {(sub_check["is_mule"]<0.5).sum():,}')
print(f'With time windows:       {(sub_check["suspicious_start"]!="").sum():,}')
print(f'\nNull checks:')
print(f'  account_id nulls: {sub_check["account_id"].isna().sum()}')
print(f'  is_mule nulls:    {sub_check["is_mule"].isna().sum()}')

# Format checks
assert len(sub_check) == 64000, f'❌ Wrong row count: {len(sub_check)}'
assert set(sub_check.columns) == {'account_id','is_mule','suspicious_start','suspicious_end'}, '❌ Wrong columns'
assert sub_check['is_mule'].between(0,1).all(), '❌ Scores out of [0,1] range'
assert sub_check['account_id'].nunique() == len(sub_check), '❌ Duplicate account_ids'

# Timestamp format check
ts_rows = sub_check[sub_check['suspicious_start'] != '']
if len(ts_rows) > 0:
    try:
        pd.to_datetime(ts_rows['suspicious_start'].head(5))
        pd.to_datetime(ts_rows['suspicious_end'].head(5))
        print(f'\nTimestamp format: ✅ Valid ISO format')
        print(f'Sample: {ts_rows["suspicious_start"].iloc[0]}')
    except Exception as e:
        print(f'\n❌ Timestamp format error: {e}')

print('\n✅ ALL CHECKS PASSED — Safe to upload submission.csv')
print('\n📤 Upload to: https://your-competition-url/submit')
print('   File: submission.csv')
print('   Column order: account_id, is_mule, suspicious_start, suspicious_end')


In [ ]:
# ── FINAL SUBMISSION ──
ta  = pd.read_parquet('test_accounts.parquet', columns=['account_id'])
sub = ta.merge(preds[['account_id','is_mule_prob']], on='account_id', how='left')
sub = sub.merge(tw[['account_id','suspicious_start','suspicious_end']], on='account_id', how='left')
# Use calibrated probability for submission (better AUC)
sub['is_mule'] = sub['is_mule_prob'].fillna(0.0).clip(0, 1).round(6)

legit_mask = sub['is_mule'] < 0.5
sub.loc[legit_mask,'suspicious_start'] = None
sub.loc[legit_mask,'suspicious_end']   = None
sub['suspicious_start'] = pd.to_datetime(sub['suspicious_start']).dt.strftime('%Y-%m-%dT%H:%M:%S').fillna('')
sub['suspicious_end']   = pd.to_datetime(sub['suspicious_end']).dt.strftime('%Y-%m-%dT%H:%M:%S').fillna('')
sub['suspicious_start'] = sub['suspicious_start'].replace('NaT','')
sub['suspicious_end']   = sub['suspicious_end'].replace('NaT','')

sub = sub[['account_id','is_mule','suspicious_start','suspicious_end']]
assert len(sub) == len(ta)
assert sub['is_mule'].between(0,1).all()

sub.to_csv('submission.csv', index=False)
print('\n🎉 submission.csv SAVED!')
print(f'Total rows:       {len(sub)}')
print(f'Predicted mules:  {(sub["is_mule"]>=0.5).sum()}')
print(f'With time window: {(sub["suspicious_start"]!="").sum()}')
sub.head(10)